#

## Part 1: Token Optimization and Token Saving

### Q1. What are the major sources of token consumption in a RAG application?

Answer:

The major sources of token consumption in a RAG application are:

1. System prompt and instructions.
2. User query.
3. Retrieved document chunks.
4. Conversation history.
5. Output generated by the LLM.

Usually, retrieved context and conversation history consume many input tokens. We can reduce token usage by removing irrelevant chunks, compressing context, and summarizing old conversation history.

### Q2. How do chunk size and top-k retrieval affect the number of input tokens sent to the LLM?

Answer:

Chunk size determines how much text is stored in each chunk, while top-k determines how many chunks are retrieved.

For example, if each chunk contains 500 tokens and top-k is 4, the retrieved context contains approximately 2,000 tokens.

If we increase chunk size or top-k, input token usage usually increases. We should choose both carefully to balance accuracy and cost.

### Q3. Suppose your retriever returns 10 chunks, each containing approximately 800 tokens. How would you reduce context size without losing important information?

Answer:

First, I would use a reranker to select the most relevant chunks. Then, I would apply contextual compression to remove irrelevant sentences.

I would also remove duplicate information and combine overlapping chunks. Finally, I would keep only the important evidence needed to answer the query.

This reduces token usage while preserving important information.

### Q4. What is contextual compression, and how can it reduce token usage in a RAG application?

Answer:

Contextual compression means reducing retrieved documents to keep only the information relevant to the user's query.

For example, if a document contains 1,000 tokens but only 150 tokens are relevant, we can pass those 150 tokens to the LLM.

This reduces input token usage, cost, and sometimes latency. However, we must preserve important facts and citation information.

### Q5. What is the difference between reducing chunk size and reducing the number of retrieved chunks?

Answer:

Reducing chunk size means retrieving smaller pieces of text. Reducing the number of retrieved chunks means retrieving fewer pieces of text.

Smaller chunks can improve relevance, but they may lose important context. Fewer chunks reduce the total context size, but they may miss useful information.

I would choose chunk size based on document structure and top-k based on the query and retrieval quality.

### Q6. How would you select dynamic top-k based on query complexity instead of using a fixed value?

Answer:

I would use dynamic top-k retrieval.

For a simple query, I might retrieve 2 or 3 chunks. For a complex query that requires multiple facts, I might retrieve 5 to 8 chunks.

I would also consider similarity scores, the number of relevant results, and a token budget. The system would stop retrieving when it has enough relevant information.

These values are examples; I would tune them using evaluation data.

### Q7. How can a reranker reduce token cost even though it introduces an additional model call?

Answer:

A reranker can reduce token cost by selecting the most relevant documents from the initial retrieved results.

For example, the retriever returns 10 chunks, but the reranker selects only the best 3. The LLM then receives less context.

Although the reranker adds some processing time and cost, it can reduce LLM input tokens and improve answer quality. I would compare the total cost and latency before and after adding it.

### Q8. What is prompt caching, and how does semantic caching differ from it?

Answer:

Prompt caching reuses the computation for repeated parts of a prompt, such as a long system prompt. It can reduce processing time and cost when supported by the model provider.

Semantic caching stores previous answers and retrieves them when a new query has a sufficiently similar meaning.

The main difference is that prompt caching reuses prompt computation, while semantic caching reuses previous results. Semantic caching must be designed carefully to avoid returning outdated or incorrect answers.

### Q9. How would you reduce repeated system prompt and conversation history tokens in a multi-turn RAG chatbot?

Answer:

I would keep the system prompt short and avoid repeating unnecessary instructions.

I would also maintain a limited conversation window and summarize older messages into a compact summary. Only relevant history would be sent to the LLM for each request.

If the model provider supports prompt caching, I would use it for repeated prompt prefixes. This reduces token usage while preserving important conversation context.

### Q10. When should conversation history be summarized rather than passed in full to the LLM?

Answer:

I would summarize conversation history when it becomes too long, increases latency, or consumes too many tokens.

I would preserve important user preferences, decisions, previous questions, and relevant facts. Recent messages would usually remain in full.

For tasks requiring exact wording or detailed history, I would retain the necessary original messages instead of relying only on a summary.

### Q11. How would you handle a large retrieved document when only one paragraph is relevant to the query?

Answer:

I would split the document into smaller chunks and retrieve the most relevant paragraphs.

Then, I would use a reranker or contextual compression to select the useful content. I would preserve the source document ID, page number, and paragraph information for citations.

This avoids sending the entire document to the LLM.

### Q12. Can aggressive context compression decrease answer accuracy? How would you find the right balance?

Answer:

Yes, aggressive compression can remove important facts, conditions, or supporting evidence. This can make the final answer incomplete or incorrect.

I would test different compression levels using a representative evaluation dataset. I would measure answer accuracy, faithfulness, citation accuracy, token usage, and latency.

I would choose the level that reduces tokens while keeping answer quality within acceptable limits.

### Q13. How would you measure token savings per request while ensuring citation accuracy and answer faithfulness do not degrade?

Answer:

First, I would record input tokens, output tokens, latency, and cost for each request before optimization.

Then, I would compare these measurements after applying techniques such as reranking, compression, and history summarization.

I would also evaluate answer faithfulness, citation correctness, and retrieval recall using a test dataset. I would deploy the optimization only if it saves tokens without causing an unacceptable drop in answer quality.

A simple formula is:

\\[ \text{Token Savings \\%} = \frac{\text{Before}-\text{After}}{\text{Before}}\times100 \\]

Here, “Before” and “After” represent the token counts for comparable requests.

## Part 2: Production-Grade RAG Latency Optimization

### Q1. Can you explain the end-to-end latency breakdown of a RAG application, from receiving a user query to delivering the final response?

Answer:

A RAG application has several stages:

1. Query processing and validation.
2. Query embedding generation.
3. Document retrieval from the vector database or search engine.
4. Reranking and context compression.
5. Prompt construction.
6. LLM response generation.
7. Response streaming or delivery to the user.

Each stage adds some latency. I would measure the time taken by every stage to identify bottlenecks and optimize the slowest components first.

### Q2. Suppose your RAG application takes 5 seconds to respond. How would you identify the performance bottleneck?

Answer:

First, I would add timing logs or distributed tracing to measure each stage separately.

For example, I would measure embedding time, retrieval time, reranking time, LLM time, and response delivery time.

If LLM generation takes 3 seconds out of the total 5 seconds, it is a major bottleneck. I would then investigate prompt size, model selection, output length, and streaming.

I would use tools such as application logs, FastAPI middleware, and OpenTelemetry to monitor latency.

### Q3. What is the difference between sequential and parallel execution in a RAG pipeline? How would you use parallelism to reduce end-to-end latency?

Answer:

In sequential execution, one operation starts after the previous operation finishes. In parallel execution, independent operations run at the same time.

For example, I can run BM25 retrieval and dense vector retrieval in parallel instead of waiting for one to finish before starting the other.

This reduces total latency because we do not need to wait for both operations one after another. However, operations with dependencies must still execute in the correct order.

### Q4. In a RAG pipeline, which operations can run in parallel, and which depend on previous stages? Explain with an example.

Answer:

Some operations can run in parallel, while others depend on previous results.

Parallel operations:

- BM25 retrieval and dense vector retrieval.
- Fetching independent metadata sources.
- Processing independent documents when the required data is already available.

Sequential operations:

- Generate the query embedding before dense vector retrieval.
- Retrieve documents before reranking those documents.
- Select and prepare context before sending it to the LLM.

For example, after receiving a query, I can run BM25 and dense retrieval in parallel. Then, I combine their results, rerank them, prepare the context, and send it to the LLM.

### Q5. How would you reduce latency when using both BM25 and dense retrieval?

Answer:

I would run BM25 and dense retrieval in parallel because they are independent retrieval methods.

Then, I would combine their results using a method such as Reciprocal Rank Fusion, also called RRF. I would avoid retrieving too many documents and use a reranker only when it provides enough value.

I would also optimize database indexes, connection pooling, and embedding generation. This can improve retrieval speed without sacrificing relevance.

### Q6. What is the difference between Time to First Token and total response latency?

Answer:

Time to First Token (TTFT) is the time between sending a request and receiving the first generated token from the LLM.

Total response latency is the time required to receive the complete response.

For example, if the first token arrives in 500 milliseconds but the full answer takes 4 seconds, the TTFT is 500 milliseconds and total response latency is 4 seconds.

TTFT is important for responsiveness, while total latency is important for completing the task.

### Q7. How can streaming responses improve perceived latency without reducing actual total generation time?

Answer:

Streaming sends the response to the user token by token instead of waiting for the complete answer.

For example, the user may see the first sentence after 500 milliseconds, even if the full answer takes 4 seconds.

Streaming improves the user experience because the user receives information earlier. However, it does not necessarily reduce the total time required to generate the answer.

### Q8. How would you use an asynchronous FastAPI endpoint to improve concurrency in a RAG service?

Answer:

I would use `async def` endpoints and asynchronous libraries for I/O operations, such as database queries, HTTP API calls, and calls to an LLM provider.

This allows the server to handle other requests while waiting for an external service to respond.

For example, while one request waits for the LLM API, FastAPI can continue processing another request.

I would also use connection pooling, timeouts, and concurrency limits to keep the service stable under load.

Example:

```
from fastapi import FastAPI
import httpxapp = FastAPI()
@app.get("/ask")
async def ask():
    async with httpx.AsyncClient() as client:
            response = await client.get(
                          "https://example.com/api"        )
    return {"status": response.status_code}
```

This example demonstrates asynchronous HTTP I/O. A real RAG endpoint would call the actual embedding, retrieval, and LLM services.

### Q9. When does asynchronous programming fail to improve latency?

Answer:

Asynchronous programming does not automatically make every operation faster.

It may not improve latency when the bottleneck is CPU-intensive processing, slow LLM generation, or a slow database query.

It can also provide little benefit when a request must wait for dependent operations to finish sequentially.

For CPU-intensive tasks, I would consider multiprocessing, worker queues, or optimized libraries. For slow external services, I would investigate the service itself rather than relying only on async programming.

### Q10. How would you design a RAG pipeline that targets sub-second Time to First Token under concurrent traffic?

Answer:

I would design the system to minimize work before the first token is generated.

My approach would be:

1. Use a fast embedding model and optimize query embedding generation.
2. Run BM25 and dense retrieval in parallel.
3. Optimize database indexes and reuse connections.
4. Limit the retrieved context using relevance filtering and a token budget.
5. Use a fast LLM with streaming enabled.
6. Use asynchronous FastAPI endpoints and appropriate concurrency limits.
7. Monitor TTFT, total latency, errors, and throughput under load.

I would test the system with realistic concurrent traffic and optimize the slowest stages.

Sub-second TTFT is a target, not a guarantee. It depends on the model provider, network conditions, retrieval speed, prompt processing, and server load.

